# 04. KNN - Implementation, training và testing

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / 'src/ml_scratch').is_dir():
    if ROOT == ROOT.parent: raise FileNotFoundError('Mở notebook trong dự án')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import display, Image


## Mục đích
mục đích: Một email ham hoặc spam, sau TF-IDF, được gán nhãn bằng những thư train nào?

Input là ma trận bước 03, không fit lại từ vựng. `fit` chỉ lưu vector train và nhãn. Với một dòng test, `euclidean_distances` tính khoảng cách tới từng email train, `nearest_indices` lấy 5 dòng gần nhất, `majority_vote` chọn nhãn. Score spam = số hàng xóm spam / 5.

Ô tiếp theo làm đúng việc đó trên một email ham và một email spam trong test: in đoạn thư, các từ TF-IDF nặng nhất, rồi bảng 5 hàng xóm. Ví dụ 2D nằm ở `docs/Algorithms/KNN.md`, không thay cho bước này. K=5 cố định, không chọn theo test.

## Một ham và một spam trong test
`fit` lưu toàn bộ vector train. Với mỗi email dưới đây, bảng đầu là từ trong chính email đó. Bảng sau là 5 email train gần nhất: `source_row` trỏ về dòng gốc, `label` là phiếu, `distance` là Euclid trên TF-IDF. Nhãn in ra là `majority_vote` của 5 phiếu đó.

In [ ]:
from ml_scratch.data import load_features
from ml_scratch.classifiers import KNNSpamClassifier
from ml_scratch.evaluation import email_terms, explain_knn

X_train, X_test, train, test, vectorizer = load_features()
print('Train', X_train.shape, 'Test', X_test.shape)
model = KNNSpamClassifier(k=5)
model.build_model()
model.fit(X_train, train.label)

for label in ['ham', 'spam']:
    position = test.label.to_numpy().tolist().index(label)
    row = test.iloc[position]
    print('---', label, 'source_row', row.name)
    print(row.clean_text[:240])
    display(email_terms(vectorizer, row.clean_text, X_test[position]))
    neighbors, vote, score = explain_knn(model, X_test[position], train)
    display(neighbors)
    print('Vote', vote, 'spam score', score)

## Chấm toàn bộ tập test
Ô dưới gọi `fit_evaluate("knn")` trên 977 email test. `fit` chỉ lưu vector train và nhãn. `predict` tính khoảng cách Euclid tới từng email train, lấy 5 hàng xóm, rồi `majority_vote`. Spam score = số hàng xóm spam / 5, nên chỉ nhận 0; 0,2; 0,4; 0,6; 0,8; 1.

K=5 cố định từ trước. Không dò K theo accuracy của ô này.

Ô ghi `artifacts/knn.json`, `outputs/knn_metrics.csv`, `knn_predictions.csv`, `knn_confusion_matrix.csv`, `knn_roc_curve.csv` và `outputs/figures/knn_test.png`. `examples()` chỉ vẽ phác họa 2D `knn_example.png`. Hình đó không thay cho bảng hàng xóm của email thật ở ô trên.

In [ ]:
from ml_scratch.evaluation import fit_evaluate, examples
metrics = fit_evaluate("knn")
examples()
metrics

## Đọc hai hình
`knn_test.png` đặt confusion matrix cạnh ROC của lần chấm này. Hàng ma trận là nhãn thật, cột là nhãn dự đoán, thứ tự ham rồi spam. ROC dùng spam score rời rạc ở trên.

`knn_example.png` là năm điểm trên mặt phẳng, một query và K=3. Đây là hình công thức, không phải email Enron.

In [ ]:
display(Image(filename=str(ROOT/"outputs/figures/knn_test.png")))
display(Image(filename=str(ROOT/"outputs/figures/knn_example.png")))

## Đọc số và handoff
Trên 977 email test, KNN có TN 665, FP 20, FN 15, TP 277. Accuracy 96,42%, precision 93,27%, recall 94,86%, F1 94,06%, FPR 2,92%, AUC 0,9882. Hai mươi ham bị gọi là spam, mười lăm spam bị sót. `knn_test.png` đặt confusion matrix cạnh ROC của các score 0; 0,2; …; 1.

Bảng hàng xóm ở trên là một ham và một spam thật, không phải năm điểm 2D. Hình `knn_example.png` chỉ là phác họa công thức, sinh khi chấm toàn bộ test.

Output: `artifacts/knn.json` (K, vector train, nhãn, TF-IDF); `outputs/knn_metrics.csv`, `knn_predictions.csv` (có `source_row`), `knn_confusion_matrix.csv`, `knn_roc_curve.csv`. Bước 06 đọc JSON và CSV này cùng bản Naive Bayes.